# Transformer Preprocessing Study

This notebook evaluates 20 different preprocessing methods for the Probabilistic Transformer model comparing against standard scaling as baseline

## Preprocessing Methods

1.  **Inverse hyperbolic sine** (arcsinh)
2.  **Mirror-log**
3.  **Probability integral transform** (uniform)
4.  **Box-cox**
5.  **Robust scaling**
6.  **MAD Scaling**
7.  **Yeo-Johnson**
8.  **Quantile transformer** (gaussian)
9.  **STL decomposition** (trend/seasonality removal)
10. **Wavelet decomposition**
11. **Variational mode decomposition**
12. **Differencing** (stationarization)
13. **Scarcity indicators** (margin feature)
14. **Residual load** (load - renewables)
15. **Cyclical time encoding**
16. **Boolean flags** (holidays, outages)
17. **Winsorization / clipping**
18. **Rolling window scaling** (adaptive)
19. **Noise injection**
20. **MixUp** (data augmentation)

Each configuration is tried 5 times to account for statistical variability

In [1]:
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler

# Fix project root to path
current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from data import DataPipeline, DataLoader
from models import ProbabilisticTransformer
from core.trainer import Trainer
from core.evaluator import Evaluator

from transformations import StandardScalingTransformation, YeoJohnsonTransformation, ArcsinhTransformation
from transformations.experimental import (
    MirrorLogTransformation, ProbabilityIntegralTransform, BoxCoxTransformation,
    MADScalingTransformation, QuantileGaussianTransformation, WinsorizationTransformation
)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

2026-02-17 00:14:53.853207: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771283693.985666  137919 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771283694.019056  137919 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771283694.269502  137919 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771283694.269523  137919 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771283694.269525  137919 computation_placer.cc:177] computation placer alr

GPUs detected: 1


## Configuration

In [2]:
RESULTS_DIR = Path(project_root) / "results" / "preprocessing_study"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "preprocessing_study_results.json"

# Base model parameters
BASE_MODEL_CONFIG = {
    "d_model": 128,
    "num_heads": 4,
    "num_layers": 2,
    "ff_dim": 256,
    "dropout": 0.1,
}

# Training settings
TRAIN_SETTINGS = {
    "batch_size": 64,
    "epochs": 20,
    "learning_rate": 1e-3,
    "patience": 5,
}

INPUT_WINDOW = 168
OUTPUT_HORIZON = 24
N_RUNS = 5

## Data preparation utilities

Helper functions for windowing, differencing and feature engineering

In [3]:
def create_windows(data, input_window, output_horizon):
    X, y = [], []
    for i in range(len(data) - input_window - output_horizon + 1):
        window = data[i:(i + input_window)]
        target = data[(i + input_window):(i + input_window + output_horizon), 0]
        # Skip windows containing NaN values
        if np.isnan(window).any() or np.isnan(target).any():
            continue
        X.append(window)
        y.append(target)
    return np.array(X), np.array(y)

def add_scarcity_margin(df):
    df = df.copy()
    # margin = generation - load
    gen_col = 'BE_Generation_Actual' if 'BE_Generation_Actual' in df.columns else None
    load_col = 'BE_Load_Actual' if 'BE_Load_Actual' in df.columns else None
    
    if gen_col and load_col:
        df['Margin'] = df[gen_col] - df[load_col]
    else:
        print("Warning: Could not calculate Margin, Missing columns.")
        df['Margin'] = 0
    return df

def add_residual_load(df):
    df = df.copy()
    # res_load = load - (solar + wind)
    load_col = 'BE_Load_Actual'
    solar_col = 'BE_Solar_Forecast'
    wind_col = 'BE_Wind_Forecast'
    
    if all(c in df.columns for c in [load_col, solar_col, wind_col]):
        df['Residual_Load'] = df[load_col] - (df[solar_col] + df[wind_col])
    else:
        print("Warning: Could not calculate ResLoad. Missing columns.")
        df['Residual_Load'] = 0
    return df

def add_noise(X, noise_level=0.01):
    noise = np.random.normal(0, noise_level, X.shape)
    return X + noise

def apply_mixup(X, y, alpha=0.2):
    indices = np.random.permutation(len(X))
    X2 = X[indices]
    y2 = y[indices]
    lam = np.random.beta(alpha, alpha, len(X))
    lam = lam.reshape(-1, 1, 1)
    X_mix = lam * X + (1 - lam) * X2
    lam_y = lam.reshape(-1, 1)
    y_mix = lam_y * y + (1 - lam_y) * y2
    return X_mix, y_mix

## Caching

In [4]:
def load_cache() -> Dict[str, Any]:
    if CACHE_FILE.exists():
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except json.JSONDecodeError:
            return {}
    return {}

def save_cache(cache: Dict[str, Any]) -> None:
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

## Experiment list

In [5]:
EXPERIMENTS = [
    "Baseline",
    "Arcsinh", 
    "MirrorLog",
    "ProbIntegralTransform",
    "BoxCox",
    "RobustScaling",
    "MADScaling",
    "YeoJohnson",
    "QuantileGaussian",
    "STLDecomposition",
    "WaveletDecomposition",
    "VMDDecomposition",
    "Differencing",
    "ScarcityIndicators",
    "ResidualLoad",
    "CyclicalEncoding",
    "BooleanFlags",
    "Winsorization",
    "RollingWindowScaling",
    "NoiseInjection",
    "MixUp"
]

In [ ]:
print("Loading Base Data...")
data_config = DataConfig(dataset_name="BE_ENTSOE")
pipeline = DataPipeline(data_config)
df_train, df_val, df_test = pipeline.get_data_splits()

cache = load_cache()

for i, exp_name in enumerate(EXPERIMENTS):
    print(f"\n[{i+1}/{len(EXPERIMENTS)}] Running: {exp_name}")
    
    if exp_name in cache:
        print("  -> Found in cache, skipping.")
        continue
        
    train_cur = df_train.copy()
    val_cur = df_val.copy()
    test_cur = df_test.copy()
    y_test_raw_prices = None
    
    if exp_name == "ScarcityIndicators":
        train_cur = add_scarcity_margin(train_cur)
        val_cur = add_scarcity_margin(val_cur)
        test_cur = add_scarcity_margin(test_cur)
    elif exp_name == "ResidualLoad":
        train_cur = add_residual_load(train_cur)
        val_cur = add_residual_load(val_cur)
        test_cur = add_residual_load(test_cur)
    elif exp_name == "Differencing":
        train_cur = train_cur.diff().dropna()
        val_cur = val_cur.diff().dropna()
        test_cur = test_cur.diff().dropna()

    # Windowing
    X_train, y_train = create_windows(train_cur.values, INPUT_WINDOW, OUTPUT_HORIZON)
    X_val, y_val = create_windows(val_cur.values, INPUT_WINDOW, OUTPUT_HORIZON)
    X_test, y_test = create_windows(test_cur.values, INPUT_WINDOW, OUTPUT_HORIZON)
    
    # Transformation
    current_transform = StandardScalingTransformation()
    
    if exp_name == "Arcsinh":
        current_transform = ArcsinhTransformation()
    elif exp_name == "MirrorLog":
        current_transform = MirrorLogTransformation()
    elif exp_name == "ProbIntegralTransform":
        current_transform = ProbabilityIntegralTransform()
    elif exp_name == "BoxCox":
        current_transform = BoxCoxTransformation()
    elif exp_name == "RobustScaling":
        from transformations.advanced import RobustScalerTransformation
        current_transform = RobustScalerTransformation()
    elif exp_name == "MADScaling":
        current_transform = MADScalingTransformation()
    elif exp_name == "YeoJohnson":
        current_transform = YeoJohnsonTransformation()
    elif exp_name == "QuantileGaussian":
        current_transform = QuantileGaussianTransformation()
    elif exp_name == "Winsorization":
        current_transform = WinsorizationTransformation()
    elif exp_name == "Differencing":
        # standard scale the differences
        current_transform = StandardScalingTransformation()
        
    # Rolling window scaling logic
    if exp_name == "RollingWindowScaling":
        means = np.mean(X_train, axis=1, keepdims=True)
        stds = np.std(X_train, axis=1, keepdims=True) + 1e-6
        X_train = (X_train - means) / stds
        X_val = (X_val - np.mean(X_val, axis=1, keepdims=True)) / (np.std(X_val, axis=1, keepdims=True) + 1e-6)
        X_test = (X_test - np.mean(X_test, axis=1, keepdims=True)) / (np.std(X_test, axis=1, keepdims=True) + 1e-6)
        
        current_transform.fit(X_train, y_train)
        # Only scale y with global stats
        _, y_train_s = current_transform.transform(X_train, y_train)
        _, y_val_s = current_transform.transform(X_val, y_val)
        _, y_test_s = current_transform.transform(X_test, y_test)
        X_train_s, X_val_s, X_test_s = X_train, X_val, X_test
        
    else:
        current_transform.fit(X_train, y_train)
        X_train_s, y_train_s = current_transform.transform(X_train, y_train)
        X_val_s, y_val_s = current_transform.transform(X_val, y_val)
        X_test_s, y_test_s = current_transform.transform(X_test, y_test)

    # Augmentation
    if exp_name == "NoiseInjection":
        X_train_s = add_noise(X_train_s)
    elif exp_name == "MixUp":
        X_train_s, y_train_s = apply_mixup(X_train_s, y_train_s)

    # Training loop
    run_metrics = []
    for r in range(N_RUNS):
        print(f"    Run {r+1}/{N_RUNS}...")
        tf.keras.backend.clear_session()
        gc.collect()
        
        model_conf = ModelConfig(**BASE_MODEL_CONFIG)
        train_conf = TrainingConfig(**TRAIN_SETTINGS)
        dc = DataConfig("BE_ENTSOE", input_window=INPUT_WINDOW, output_horizon=OUTPUT_HORIZON)
        
        exp_conf = ExperimentConfig(
            name=f"{exp_name}_run{r}",
            data_config=dc,
            model_config=model_conf,
            training_config=train_conf,
            head_type="johnson_su"
        )
        
        model = ProbabilisticTransformer(exp_conf)
        trainer = Trainer(exp_conf)
        
        st = time.time()
        try:
            trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s)
        except Exception as e:
            print(f"    Training Failed: {e}")
            continue
            
        fit_time = time.time() - st
        
        # evaluate
        if exp_name == "Differencing":
            # Reconstruction logic
            y_pred_params = model.keras_model.predict(X_test_s, verbose=0)
            means_diff_scaled = model.head.mean(y_pred_params.reshape(-1, y_pred_params.shape[-1])).reshape(y_pred_params.shape[0], -1)
            
            # Inverse scale to get raw differences
            _, means_diff = current_transform.inverse_transform(X=None, y=means_diff_scaled)
            
            X_test_raw_full, y_test_raw_full = create_windows(df_test.values, INPUT_WINDOW, OUTPUT_HORIZON)
            shift = 1
            min_len = min(len(means_diff), len(y_test_raw_full) - shift)
            
            last_known_prices = X_test_raw_full[shift : shift+min_len, -1, 0].reshape(-1, 1)
            targets_raw = y_test_raw_full[shift : shift+min_len, 0]
            if OUTPUT_HORIZON > 1:
                targets_raw = y_test_raw_full[shift : shift+min_len]
                
            # Reconstruct: last + CumSum(Diffs)
            preds_recon = np.cumsum(means_diff[:min_len], axis=1) + last_known_prices
            
            # Calc MAE
            mae = np.mean(np.abs(preds_recon - targets_raw))
            rmse = np.sqrt(np.mean((preds_recon - targets_raw)**2))
            
            metrics = {"MAE": mae, "RMSE": rmse, "PICP": 0, "MPIW": 0, "CRPS": 0}
            
        else:
            evaluator = Evaluator(model, current_transform)
            metrics = evaluator.evaluate(X_test_s, y_test)
            
        metrics['fit_time'] = fit_time
        run_metrics.append(metrics)
        
    if not run_metrics: continue
    
    avg_metrics = {}
    for k in run_metrics[0].keys():
        avg_metrics[k] = float(np.mean([m[k] for m in run_metrics]))
    
    cache[exp_name] = avg_metrics
    save_cache(cache)
    print(f"    MAE: {avg_metrics.get('MAE', 0):.4f}")

Loading Base Data...

[1/21] Running: Baseline
  -> Found in cache, skipping.

[2/21] Running: Arcsinh
  -> Found in cache, skipping.

[3/21] Running: MirrorLog
  -> Found in cache, skipping.

[4/21] Running: ProbIntegralTransform
  -> Found in cache, skipping.

[5/21] Running: BoxCox
  -> Found in cache, skipping.

[6/21] Running: RobustScaling
  -> Found in cache, skipping.

[7/21] Running: MADScaling
  -> Found in cache, skipping.

[8/21] Running: YeoJohnson
  -> Found in cache, skipping.

[9/21] Running: QuantileGaussian
  -> Found in cache, skipping.

[10/21] Running: STLDecomposition
  -> Found in cache, skipping.

[11/21] Running: WaveletDecomposition
  -> Found in cache, skipping.

[12/21] Running: VMDDecomposition
  -> Found in cache, skipping.

[13/21] Running: Differencing
  -> Found in cache, skipping.

[14/21] Running: ScarcityIndicators
  -> Found in cache, skipping.

[15/21] Running: ResidualLoad
  -> Found in cache, skipping.

[16/21] Running: CyclicalEncoding
  -> Foun

In [7]:
results = []
for k, v in cache.items():
    row = {"Method": k, **v}
    results.append(row)
    
df_res = pd.DataFrame(results)
if not df_res.empty:
    if "MAE" in df_res.columns:
        df_res = df_res.sort_values("MAE")
    display(df_res)
    df_res.to_csv(RESULTS_DIR / "summary.csv", index=False)
else:
    print("No results.")

,Method,MAE,MSE,RMSE,PICP,MPIW,IntervalScore,CRPS,fit_time
14,ResidualLoad,20.433684,778.680185,27.876807,0.918019,99.533215,154.401008,14.981954,47.795368
10,WaveletDecomposition,20.542791,767.694254,27.687630,0.914576,99.288002,151.693919,15.227782,47.991163
20,MixUp,20.583503,780.735532,27.936155,0.922798,100.849922,151.263851,15.336258,46.444877
15,CyclicalEncoding,20.729084,781.287809,27.932111,0.933030,109.886455,151.490162,15.082650,46.379416
19,NoiseInjection,20.769305,792.242780,28.131911,0.917169,103.233926,155.879408,15.285351,48.352200
13,ScarcityIndicators,20.769882,805.777351,28.378272,0.923951,102.819197,154.871423,15.431078,46.743259
16,BooleanFlags,20.836802,798.597969,28.258704,0.929300,106.399241,151.926341,15.286710,47.524804
7,YeoJohnson,21.208243,807.957106,28.410353,0.935624,120.301648,158.463246,15.672012,45.820978
0,Baseline,21.273060,832.485895,28.808033,0.912382,101.908289,163.117156,15.790088,46.872009
11,VMDDecomposition,21.422027,827.843569,28.768604,0.937160,111.786823,152.207509,15.597311,46.433834
